In [1]:
getwd()
setwd("/liulab/galib/dlbcl_manuscript/")
library(rBCS)
library(tidyverse)
library(Seurat)
library(harmony)
library(viridis)
library(RColorBrewer)
library(Polychrome)
PurpleAndYellow()
library(ComplexHeatmap)
library(devtools)
library(presto)
library(dplyr)
library(ggplot2)
library(ggpubr)
library(readxl)
source("./scripts/scplot.R")
library(DESeq2)
set.seed(123)

[1] "/liulab/galib/dlbcl_manuscript/scripts/Revision"

Warning message:
“package ‘rBCS’ was built under R version 4.1.3”
Warning message:
“package ‘tidyverse’ was built under R version 4.1.3”
── Attaching packages ─────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.6      ✔ purrr   0.3.4 
✔ tibble  3.1.8      ✔ dplyr   1.0.10
✔ tidyr   1.2.0      ✔ stringr 1.4.1 
✔ readr   2.1.2      ✔ forcats 0.5.1 

Warning message:
“package ‘tidyr’ was built under R version 4.1.2”
Warning message:
“package ‘readr’ was built under R version 4.1.2”
Warning message:
“package ‘forcats’ was built under R version 4.1.3”
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()

Attaching SeuratObject

Attaching sp

Warning message:
“package ‘harmony’ was built under R version 4.1.3”
Loading required package: Rcpp

Warning message:
“package ‘Rcpp’ was built under R version 4.1.2”
Loading required package: viridisLite

Warning message:
“pack

[1] "#FF00FF" "#F400F4" "#EA00EA" "#DF00DF" "#D500D5" "#CA00CA" "#BF00BF"
 [8] "#B500B5" "#AA00AA" "#9F009F" "#950095" "#8A008A" "#800080" "#750075"
[15] "#6A006A" "#600060" "#550055" "#4A004A" "#400040" "#350035" "#2B002B"
[22] "#200020" "#150015" "#0B000B" "#000000" "#000000" "#0B0B00" "#151500"
[29] "#202000" "#2B2B00" "#353500" "#404000" "#4A4A00" "#555500" "#606000"
[36] "#6A6A00" "#757500" "#808000" "#8A8A00" "#959500" "#9F9F00" "#AAAA00"
[43] "#B5B500" "#BFBF00" "#CACA00" "#D4D400" "#DFDF00" "#EAEA00" "#F4F400"
[50] "#FFFF00"

Warning message:
“package ‘ComplexHeatmap’ was built under R version 4.1.3”
Loading required package: grid

ComplexHeatmap version 2.10.0
Bioconductor page: http://bioconductor.org/packages/ComplexHeatmap/
Github page: https://github.com/jokergoo/ComplexHeatmap
Documentation: http://jokergoo.github.io/ComplexHeatmap-reference

If you use it in published research, please cite:
Gu, Z. Complex heatmaps reveal patterns and correlations in multidimensional 
  genomic data. Bioinformatics 2016.

The new InteractiveComplexHeatmap package can directly export static 
complex heatmaps into an interactive Shiny app with zero effort. Have a try!

This message can be suppressed by:
  suppressPackageStartupMessages(library(ComplexHeatmap))


Loading required package: usethis

Warning message:
“package ‘presto’ was built under R version 4.1.3”
Loading required package: data.table


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


The

In [33]:
run_de_pseudobulk_deseq2 <- function(obj, outdir, prefix) {

    set.seed(123)

    raw_counts <- AggregateExpression(obj, group.by = "orig.ident", slot = "counts")
    count_matrix <- round(as.matrix(raw_counts$RNA))
    
    
    keep_genes <- rowSums(count_matrix >= 3) >= 3
    count_matrix <- count_matrix[keep_genes, ]

    coldata <- obj@meta.data %>%
        distinct(orig.ident, genotype) %>%
        as.data.frame(row.names = NULL)
    rownames(coldata) <- coldata$orig.ident
    coldata$orig.ident <- NULL
    coldata <- coldata[colnames(count_matrix), , drop = FALSE]
    coldata$genotype <- factor(coldata$genotype, levels = c("Bcl6tg/+", "CD70-/-;Bcl6tg/+"))

    dds <- DESeqDataSetFromMatrix(countData = count_matrix,
                                  colData = coldata,
                                  design = ~ genotype)
    dds <- DESeq(dds)
    res <- results(dds, contrast = c("genotype", "CD70-/-;Bcl6tg/+", "Bcl6tg/+"))

    markers <- as.data.frame(res) %>%
        rownames_to_column("feature") %>%
        dplyr::rename(logFC = log2FoldChange, p = pvalue) %>%
        dplyr::mutate(padj_BH = p.adjust(p, method = "BH"))  %>% 
        dplyr::select(feature, logFC, p, padj_BH) %>%
        arrange(padj_BH)


    write_tsv(markers, paste0(outdir, prefix, "_all_markers.tsv"))

    # Volcano plot
    markers$neg_log10_padj <- -log10(markers$padj_BH)
    markers$sig <- case_when(
        markers$padj_BH <= 0.05 & markers$logFC >= 1.0  ~ "Up",
        markers$padj_BH <= 0.05 & markers$logFC <= -1.0 ~ "Down",
        TRUE ~ "NS"
    )

    top_genes <- markers %>%
        filter(sig != "NS") %>%
        arrange(padj_BH) %>%
        slice_head(n = 20)

    p_volcano <- ggplot(markers, aes(x = logFC, y = neg_log10_padj, color = sig)) +
        geom_point(alpha = 0.5, size = 1) +
        scale_color_manual(values = c("Up" = "#E64B35", "Down" = "#4DBBD5", "NS" = "grey70")) +
        geom_vline(xintercept = c(-1.0, 1.0), linetype = "dashed", color = "black") +
        geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black") +
        ggrepel::geom_text_repel(
            data = top_genes,
            aes(label = feature),
            size = 3, max.overlaps = 20, color = "black"
        ) +
        labs(
            title = paste0(prefix, ":\nVolcano plot padj <= 0.05 abs(logFC) >= 1.0"),
            x = "log2 Fold Change",
            y = "-log10(adjusted p-value)",
            color = NULL
        ) +
        theme_classic(base_size = 14)

    ggsave(paste0(outdir, prefix, "_volcano_plot.pdf"), p_volcano, width = 7, height = 5)

    # Histograms
    p_padj <- ggplot(markers, aes(x = padj_BH)) +
        geom_histogram(bins = 30, fill = "#4DBBD5", color = "black") +
        geom_vline(xintercept = 0.05, linetype = "dashed", color = "red") +
        labs(
            title = paste0(prefix, ":\nAdjusted p-value distribution"),
            x = "Adjusted p-value",
            y = "Number of genes"
        ) + theme_classic(base_size = 14)

    p_logfc <- ggplot(markers, aes(x = logFC)) +
        geom_histogram(bins = 30, fill = "#E64B35", color = "black") +
        geom_vline(xintercept = c(-1.0, 1.0), linetype = "dashed", color = "blue") +
        labs(
            title = paste0(prefix, ":\nlog2 Fold Change distribution"),
            x = "log2 Fold Change",
            y = "Number of genes"
        ) + theme_classic(base_size = 14)

    ggsave(paste0(outdir, prefix, "_padj_histogram.pdf"), p_padj, width = 5, height = 4)
    ggsave(paste0(outdir, prefix, "_logFC_histogram.pdf"), p_logfc, width = 5, height = 4)

    # Significant markers
    markers_sig <- markers %>% filter(padj_BH <= 0.05, abs(logFC) >= 1.0) %>% arrange(padj_BH)
    write_tsv(markers_sig, paste0(outdir, prefix, "_significant_markers_padj_0.05_abslogFC_1.0.tsv"))

    up_in_double <- markers_sig %>%
        filter(logFC > 0) %>%
        arrange(padj_BH) %>%
        slice_head(n = 10)

    down_in_double <- markers_sig %>%
        filter(logFC < 0) %>%
        arrange(padj_BH) %>%
        slice_head(n = 10)

    genelist <- c(up_in_double$feature, down_in_double$feature) %>% unique()

    if (length(genelist) == 0) {
        message("No significant DE genes for ", prefix)
    } else {
        write_tsv(up_in_double, paste0(outdir, prefix, "_top10_up_regulated_in_double.tsv"))
        write_tsv(down_in_double, paste0(outdir, prefix, "_top10_down_regulated_in_double.tsv"))

        p <- DotPlot(object = obj, features = genelist, group.by = "genotype", scale = TRUE) +
            theme(axis.text.x = element_text(angle = 90, hjust = 1)) +
            labs(title = prefix, y = "genotype")
        ggsave(paste0(outdir, prefix, "_top10_up_and_down_regulated_genes_in_double.pdf"), p, width = 7, height = 4)

        p <- VlnPlot(obj, features = genelist, group.by = "genotype", ncol = 10, pt.size = 0.2)
        ggsave(paste0(outdir, prefix, "_top10_up_and_down_regulated_genes_in_double_vlnplot.pdf"), p, width = 12, height = 5)

        p <- VlnPlot(obj, features = genelist, group.by = "orig.ident", split.by = "genotype",
                      pt.size = 0.2, stack = TRUE, sort = TRUE, flip = TRUE)
        ggsave(paste0(outdir, prefix, "_top10_up_and_down_regulated_genes_in_double_vlnplot_by_sample.pdf"), p, width = 14, height = 8)
    }
}

In [3]:
cd3_pos_cd8_neg<- readRDS("./data/objects/cd3_pos_cd8_neg_final.rds")

In [4]:
# Diff analysis of CD4 CTLs (C1, 4a, 6, 13) in Bcl6 vs double mice at: 6 mos; 14 mos; 18 mos; 
# sick; 14+18+sick; all time points followed by GSEA (and volcano plots)
cd3_pos_cd8_neg$genotype  %>% table()
cd3_pos_cd8_neg$age  %>% table()

.
        Bcl6tg/+          CD70-/- CD70-/-;Bcl6tg/+               WT 
           21898             9504            15675             7408 

.
14mos 18mos  6mos  sick 
10986 17595 12277 13627 

In [34]:
# Downsample CD4 CTLs in Bck6 and double mice,all time points

prefix = 'cd4_ctl_de_all_timepoints'

cd4_ctls<- 
    cd3_pos_cd8_neg@meta.data  %>% 
    filter(seurat_clusters %in% c(1, '4a', 6, 13), 
    genotype %in% c("Bcl6tg/+", "CD70-/-;Bcl6tg/+"))  %>% rownames()

cd3_pos_cd8_neg_ctls_tumor<- subset(cd3_pos_cd8_neg, cells = cd4_ctls)

cd3_pos_cd8_neg_ctls_tumor@meta.data  %>% distinct(genotype, orig.ident)  %>% dplyr::count(genotype)
run_de_pseudobulk_deseq2(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/deseq2/', prefix = prefix) 

genotype,n
<chr>,<int>
Bcl6tg/+,25
CD70-/-;Bcl6tg/+,28


converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R.

In [35]:
prefix = 'cd4_ctl_de_6mos'

cd4_ctls<- 
    cd3_pos_cd8_neg@meta.data  %>% 
    filter(seurat_clusters %in% c(1, '4a', 6, 13), 
    genotype %in% c("Bcl6tg/+", "CD70-/-;Bcl6tg/+"),
    age %in% c("6mos"))  %>% rownames()

length(cd4_ctls)

cd3_pos_cd8_neg_ctls_tumor<- subset(cd3_pos_cd8_neg, cells = cd4_ctls)

# run_de_pseudobulk(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/', prefix = prefix) 
cd3_pos_cd8_neg_ctls_tumor@meta.data  %>% distinct(genotype, orig.ident)  %>% dplyr::count(genotype)
run_de_pseudobulk_deseq2(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/deseq2/', prefix = prefix) 

[1] 704

genotype,n
<chr>,<int>
Bcl6tg/+,6
CD70-/-;Bcl6tg/+,6


converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R.

In [38]:
# Downsample CD4 CTLs in Bck6 and double mice, 14mos

prefix = 'cd4_ctl_de_14mos'

cd4_ctls<- 
    cd3_pos_cd8_neg@meta.data  %>% 
    filter(seurat_clusters %in% c(1, '4a', 6, 13), 
    genotype %in% c("Bcl6tg/+", "CD70-/-;Bcl6tg/+"),
    age %in% c("14mos"))  %>% rownames()

length(cd4_ctls)

cd3_pos_cd8_neg_ctls_tumor<- subset(cd3_pos_cd8_neg, cells = cd4_ctls)

# run_de_pseudobulk(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/', prefix = prefix) 
cd3_pos_cd8_neg_ctls_tumor@meta.data  %>% distinct(genotype, orig.ident)  %>% dplyr::count(genotype)
run_de_pseudobulk_deseq2(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/deseq2/', prefix = prefix) 

[1] 3684

genotype,n
<chr>,<int>
Bcl6tg/+,5
CD70-/-;Bcl6tg/+,7


converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R.

In [39]:
# Downsample CD4 CTLs in Bck6 and double mice, 18mos
prefix = 'cd4_ctl_de_18mos'

cd4_ctls<- 
    cd3_pos_cd8_neg@meta.data  %>% 
    filter(seurat_clusters %in% c(1, '4a', 6, 13), 
    genotype %in% c("Bcl6tg/+", "CD70-/-;Bcl6tg/+"),
    age %in% c("18mos"))  %>% rownames()

length(cd4_ctls)

cd3_pos_cd8_neg_ctls_tumor<- subset(cd3_pos_cd8_neg, cells = cd4_ctls)


# run_de_pseudobulk(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/', prefix = prefix) 
cd3_pos_cd8_neg_ctls_tumor@meta.data  %>% distinct(genotype, orig.ident)  %>% dplyr::count(genotype)
run_de_pseudobulk_deseq2(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/deseq2/', prefix = prefix) 

[1] 5985

genotype,n
<chr>,<int>
Bcl6tg/+,8
CD70-/-;Bcl6tg/+,7


converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R.

In [40]:
# Downsample CD4 CTLs in Bck6 and double mice, sick

prefix = 'cd4_ctl_de_sick'

cd4_ctls<- 
    cd3_pos_cd8_neg@meta.data  %>% 
    filter(seurat_clusters %in% c(1, '4a', 6, 13), 
    genotype %in% c("Bcl6tg/+", "CD70-/-;Bcl6tg/+"),
    age %in% c("sick"))  %>% rownames()

length(cd4_ctls)

cd3_pos_cd8_neg_ctls_tumor<- subset(cd3_pos_cd8_neg, cells = cd4_ctls)

# run_de_pseudobulk(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/', prefix = prefix) 
cd3_pos_cd8_neg_ctls_tumor@meta.data  %>% distinct(genotype, orig.ident)  %>% dplyr::count(genotype)
run_de_pseudobulk_deseq2(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/deseq2/', prefix = prefix) 

[1] 2869

genotype,n
<chr>,<int>
Bcl6tg/+,6
CD70-/-;Bcl6tg/+,8


converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R.

In [41]:
# Downsample CD4 CTLs in Bck6 and double mice, 

prefix = 'cd4_ctl_de_14_18_sick'

cd4_ctls<- 
    cd3_pos_cd8_neg@meta.data  %>% 
    filter(
        seurat_clusters %in% c(1, '4a', 6, 13), 
        genotype %in% c("Bcl6tg/+", "CD70-/-;Bcl6tg/+"),
        age %in% c("14mos", "18mos", "sick"))  %>% rownames()

length(cd4_ctls)

cd3_pos_cd8_neg_ctls_tumor<- subset(cd3_pos_cd8_neg, cells = cd4_ctls)


# run_de_pseudobulk(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/', prefix = prefix) 
cd3_pos_cd8_neg_ctls_tumor@meta.data  %>% distinct(genotype, orig.ident)  %>% dplyr::count(genotype)
run_de_pseudobulk_deseq2(cd3_pos_cd8_neg_ctls_tumor, outdir = './data/revision/pseudobulk_de/deseq2/', prefix = prefix) 

[1] 12538

genotype,n
<chr>,<int>
Bcl6tg/+,19
CD70-/-;Bcl6tg/+,22


converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R.